# Stage 4A — Baseline Detector Inference

This notebook:
1. Loads `face_manifest.csv`
2. Runs a baseline deepfake detector on face crops
3. Aggregates frame-level scores to video-level scores
4. Saves:
   - `frame_detector_scores.csv`
   - `video_detector_scores.csv`

Use this notebook **before** the merge + experiments notebook.

## Assumption

This notebook assumes you already have a trained or pretrained **binary deepfake detector checkpoint**.

Typical label convention:
- `0` = real
- `1` = fake

You only need to adapt:
- `MODEL_NAME`
- `CHECKPOINT_PATH`
- checkpoint loading logic if your format differs

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoFeatureExtractor, AutoModelForImageClassification

In [10]:
# =========================
# Configuration
# =========================
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()

SUBSET_NAME = "pilot"   # "pilot" or "final"

FACE_MANIFEST_PATH = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_face_manifest.csv"
FRAME_SCORES_OUT   = PROJECT_ROOT / "datasets/detector_processed" / f"{SUBSET_NAME}_frame_detector_scores.csv"
VIDEO_SCORES_OUT   = PROJECT_ROOT / "datasets/detector_processed" / f"{SUBSET_NAME}_detector_scores.csv"

# ── Model ────────────────────────────────────────────────────────────────────
# Recommended: HuggingFace ViT-based deepfake detector (no registration needed)
HF_MODEL_ID = "dima806/deepfake_vs_real_image_detection"

# Academic standard alternative: Xception from FaceForensics++
# To use: set HF_MODEL_ID = None and fill in below
# MODEL_NAME    = "xception"
# CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "xception-hq.pth"

BATCH_SIZE  = 32
NUM_WORKERS = 2
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# Aggregation
VIDEO_SCORE_MODE = "mean"   # "mean", "max", "topk_mean"
TOPK = 10

(PROJECT_ROOT / "datasets/detector_processed").mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FACE_MANIFEST_PATH:", FACE_MANIFEST_PATH)
print("HF_MODEL_ID:", HF_MODEL_ID)
print("DEVICE:", DEVICE)


PROJECT_ROOT: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness
FACE_MANIFEST_PATH: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/pilot_face_manifest.csv
HF_MODEL_ID: dima806/deepfake_vs_real_image_detection
DEVICE: cuda


## Load face manifest

In [ ]:
assert FACE_MANIFEST_PATH.exists(), f"Face manifest not found: {FACE_MANIFEST_PATH}"
face_df = pd.read_csv(FACE_MANIFEST_PATH)
print(f"Loaded {len(face_df):,} face rows from {FACE_MANIFEST_PATH}")
display(face_df.head())

Loaded 11,926 face rows from /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/pilot_face_manifest.csv


,video_id,frame_id,timestamp_sec,label,split,manipulation_family,manipulation_type,identity,source_subset,video_path,frame_path,face_id,face_path,bbox_x1,bbox_y1,bbox_x2,bbox_y2,det_score,face_width,face_height
0,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000000,0.0,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000000_face0,/home/n.salikhova@innopolis.university/deepfak...,314,50,479,215,1.0,165,165
1,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000001,0.2,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000001_face0,/home/n.salikhova@innopolis.university/deepfak...,317,55,479,217,1.0,162,162
2,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000002,0.4,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000002_face0,/home/n.salikhova@innopolis.university/deepfak...,336,78,495,237,1.0,159,159
3,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000003,0.6,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000003_face0,/home/n.salikhova@innopolis.university/deepfak...,344,85,506,247,1.0,162,162
4,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000004,0.8,real,train,NaN,YouTube-real,NaN,YouTube-real,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,YouTube-real__00246__bc0e9215_000004_face0,/home/n.salikhova@innopolis.university/deepfak...,353,89,509,245,1.0,156,156


## Dataset

In [11]:
class FaceDetectorDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_extractor):
        self.df = df.reset_index(drop=True).copy()
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row["face_path"]).convert("RGB")
        return {
            "image": img,
            "video_id":           str(row["video_id"]),
            "frame_id":           str(row["frame_id"]),
            "face_id":            str(row["face_id"]),
            "face_path":          str(row["face_path"]),
            "label":              str(row.get("label", "")),
            "split":              str(row.get("split", "")),
            "timestamp_sec":      float(row["timestamp_sec"]) if pd.notna(row.get("timestamp_sec")) else float("nan"),
            "manipulation_family": str(row.get("manipulation_family", "")),
            "manipulation_type":   str(row.get("manipulation_type", "")),
        }


def collate_fn(batch):
    """Custom collate: process images through feature_extractor as a batch."""
    return batch   # keep as list of dicts; we batch images inside inference


dataset = FaceDetectorDataset(face_df, feature_extractor=None)  # extractor set after model load
print("Dataset size:", len(dataset))


Dataset size: 11926


## Build model

This version uses `timm`.  
If your checkpoint was trained with another architecture, adapt this cell.

In [13]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained(HF_MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(HF_MODEL_ID).to(DEVICE).eval()

# Determine which output index corresponds to "Fake"
fake_label_idx = next(
    (i for i, lbl in model.config.id2label.items() if lbl.strip().lower() == "fake"),
    1,  # fallback: index 1
)
print("Model labels :", model.config.id2label)
print("Fake label idx:", fake_label_idx)
print("Model loaded  :", HF_MODEL_ID)

# Attach processor to dataset and build DataLoader
dataset.feature_extractor = processor
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=lambda x: x,   # list-of-dicts; images batched inside inference fn
)
print(f"DataLoader ready — {len(loader)} batches")


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 5765.21it/s]


Model labels : {0: 'Real', 1: 'Fake'}
Fake label idx: 1
Model loaded  : dima806/deepfake_vs_real_image_detection
DataLoader ready — 373 batches


## Inference

In [14]:
@torch.no_grad()
def run_detector_inference(
    model: nn.Module,
    processor,
    loader: DataLoader,
    fake_idx: int = 1,
) -> pd.DataFrame:
    rows = []

    for batch in tqdm(loader, total=len(loader), desc="Running detector"):
        images = [item["image"] for item in batch]

        inputs = processor(images=images, return_tensors="pt").to(DEVICE)
        logits = model(**inputs).logits          # (B, num_classes)
        probs  = torch.softmax(logits, dim=-1)   # (B, num_classes)
        fake_probs = probs[:, fake_idx].cpu().numpy()

        for i, item in enumerate(batch):
            rows.append({
                "video_id":            item["video_id"],
                "frame_id":            item["frame_id"],
                "face_id":             item["face_id"],
                "face_path":           item["face_path"],
                "label":               item["label"],
                "split":               item["split"],
                "timestamp_sec":       item["timestamp_sec"],
                "manipulation_family": item["manipulation_family"],
                "manipulation_type":   item["manipulation_type"],
                "detector_score":      float(fake_probs[i]),
                "detector_pred":       int(fake_probs[i] >= 0.5),
            })

    return pd.DataFrame(rows)


frame_scores_df = run_detector_inference(model, processor, loader, fake_label_idx)
print(f"Generated {len(frame_scores_df):,} frame-level detector scores")
display(frame_scores_df.head())

frame_scores_df.to_csv(FRAME_SCORES_OUT, index=False)
print("Saved:", FRAME_SCORES_OUT)


Running detector: 100%|██████████| 373/373 [00:45<00:00,  8.20it/s]


Generated 11,926 frame-level detector scores


,video_id,frame_id,face_id,face_path,label,split,timestamp_sec,manipulation_family,manipulation_type,detector_score,detector_pred
0,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000000,YouTube-real__00246__bc0e9215_000000_face0,/home/n.salikhova@innopolis.university/deepfak...,real,train,0.0,nan,YouTube-real,0.052394,0
1,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000001,YouTube-real__00246__bc0e9215_000001_face0,/home/n.salikhova@innopolis.university/deepfak...,real,train,0.2,nan,YouTube-real,0.012909,0
2,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000002,YouTube-real__00246__bc0e9215_000002_face0,/home/n.salikhova@innopolis.university/deepfak...,real,train,0.4,nan,YouTube-real,0.823538,1
3,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000003,YouTube-real__00246__bc0e9215_000003_face0,/home/n.salikhova@innopolis.university/deepfak...,real,train,0.6,nan,YouTube-real,0.061806,0
4,YouTube-real__00246__bc0e9215,YouTube-real__00246__bc0e9215_000004,YouTube-real__00246__bc0e9215_000004_face0,/home/n.salikhova@innopolis.university/deepfak...,real,train,0.8,nan,YouTube-real,0.943719,1


Saved: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/detector_processed/pilot_frame_detector_scores.csv


## Aggregate to video-level scores

In [15]:
def aggregate_video_scores(df: pd.DataFrame, mode: str = "mean", topk: int = 10) -> pd.DataFrame:
    rows = []

    for video_id, grp in tqdm(df.groupby("video_id"), desc="Aggregating video scores"):
        scores = grp["detector_score"].to_numpy()

        if mode == "mean":
            video_score = float(np.mean(scores))
        elif mode == "max":
            video_score = float(np.max(scores))
        elif mode == "topk_mean":
            k = min(topk, len(scores))
            top_scores = np.sort(scores)[-k:]
            video_score = float(np.mean(top_scores))
        else:
            raise ValueError(f"Unknown aggregation mode: {mode}")

        rows.append({
            "video_id": video_id,
            "label": grp["label"].iloc[0],
            "split": grp["split"].iloc[0],
            "manipulation_family": grp["manipulation_family"].iloc[0],
            "manipulation_type": grp["manipulation_type"].iloc[0],
            "n_face_frames": len(grp),
            "video_score_mode": mode,
            "detector_score": video_score,
            "detector_pred": int(video_score >= 0.5),
        })

    return pd.DataFrame(rows)

video_scores_df = aggregate_video_scores(frame_scores_df, mode=VIDEO_SCORE_MODE, topk=TOPK)
print(f"Generated {len(video_scores_df):,} video-level detector scores")
display(video_scores_df.head())

video_scores_df.to_csv(VIDEO_SCORES_OUT, index=False)
print("Saved:", VIDEO_SCORES_OUT)

Aggregating video scores: 100%|██████████| 200/200 [00:00<00:00, 3871.16it/s]

Generated 200 video-level detector scores


,video_id,label,split,manipulation_family,manipulation_type,n_face_frames,video_score_mode,detector_score,detector_pred
0,Celeb-real__id0_0007__4f22f911,real,val,nan,Celeb-real,79,mean,0.295776,0
1,Celeb-real__id10_0007__841eee79,real,test,nan,Celeb-real,83,mean,0.867769,1
2,Celeb-real__id10_0009__00911e7c,real,test,nan,Celeb-real,90,mean,0.394559,0
3,Celeb-real__id11_0003__52787b89,real,train,nan,Celeb-real,53,mean,0.860321,1
4,Celeb-real__id12_0006__637d998d,real,train,nan,Celeb-real,52,mean,0.953136,1


Saved: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/detector_processed/pilot_detector_scores.csv


## Quick checks

In [16]:
print("Frame-level rows:", len(frame_scores_df))
print("Video-level rows:", len(video_scores_df))
display(video_scores_df[["label", "split"]].value_counts().rename("count").reset_index())

Frame-level rows: 11926
Video-level rows: 200


,label,split,count
0,real,train,67
1,fake,train,64
2,real,test,21
3,fake,test,19
4,fake,val,17
5,real,val,12


## Next

After this notebook:
1. Merge
   - `manifest`
   - `video_emotion_features.csv`
   - `detector_scores.csv`
2. Run:
   - baseline metrics
   - emotion analysis
   - fusion model
   - ablation